# CivicVault / CivicDAO — Fast Prompting POC

> POC de priorización y gobierno de proyectos comunitarios usando **Fast Prompting**.
>
> Técnicas: *instrucciones comprimidas*, *salida estructurada (JSON)*, *few-shot mínimo*,
> *self-check en la misma llamada*, *batching*, *cadena de densidad* para resúmenes, y *modo MOCK* para desarrollar sin API.


## 1. Setup

- Si tienes `OPENAI_API_KEY`, el notebook llamará a la API.
- Si no, caerá en **modo MOCK** con reglas simples para que todo funcione.


In [ ]:

import os, json, re, math, hashlib, random
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

# Intentar cargar cliente de OpenAI, pero no es obligatorio para correr en modo MOCK
USE_MOCK = False
try:
    import openai  # type: ignore
    if not os.getenv("OPENAI_API_KEY"):
        USE_MOCK = True
    else:
        openai.api_key = os.getenv("OPENAI_API_KEY")
except Exception:
    USE_MOCK = True

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")  # editable
SEED = 42
random.seed(SEED)

DATA_PATH = "data/proposals_sample.json"
with open(DATA_PATH, "r", encoding="utf-8") as f:
    PROPOSALS = json.load(f)

len(PROPOSALS), PROPOSALS[0]["titulo"]


## 2. Esquema y criterios

**Salida por propuesta (JSON):**
```json
{
  "id": "P-001",
  "categoria": "Sostenibilidad|Educación|Tecnología|Cultura|Otro",
  "resumen_120w": "…",
  "duracion_meses": 0,
  "coste_eur": 0,
  "riesgos_clave": ["…"],
  "impacto_esperado": "…",
  "puntuacion": {
    "impacto": 0,
    "viabilidad": 0,
    "costo_beneficio": 0,
    "riesgo": 0,
    "alineacion": 0,
    "total": 0
  },
  "racional_voto": "…",
  "banderas_riesgo": ["si faltan datos, listarlas aquí"],
  "valido": true
}
```
Criterios puntúan **0–5**; `total` es la media redondeada a una decimal.


In [ ]:

SYSTEM_FAST = (
    "Eres un asistente de gobernanza para una DAO cívica. "
    "Transforma propuestas en salidas JSON válidas, concisas y auditables. "
    "No inventes datos: si faltan, deja null o banderas de riesgo."
)

TEMPLATE_COMPACT = (
    "TAREA: Para cada propuesta, devuelve una lista JSON con objetos validados. "
    "ESQUEMA: {schema}. "
    "REGLAS: "
    "1) No texto fuera de JSON. 2) Duración en meses (si no aparece, null). "
    "3) Puntuación 0–5 y 'total' = media. 4) 'resumen_120w' en <=120 palabras, estilo neutro, "
    "   usando cadena de densidad: empieza amplio y condensa manteniendo entidades clave. "
    "5) 'racional_voto' en 1–2 frases. "
    "6) 'banderas_riesgo' si faltan datos críticos. "
)

FEW_SHOT = [
    {
        "id":"DEMO",
        "titulo":"Ejemplo de Taller de Bicis",
        "descripcion":"Espacio para reparar bicis con voluntariado. Duración 6 meses. Bajos riesgos.",
        "monto_objetivo_eur": 5000,
        "categoria_sugerida": "Tecnología",
        "estado": "PROPUESTA"
    }
]


In [ ]:

def schema_dict():
    return {
        "id":"str",
        "categoria":"str",
        "resumen_120w":"str",
        "duracion_meses":"int|null",
        "coste_eur":"int",
        "riesgos_clave":"list[str]",
        "impacto_esperado":"str",
        "puntuacion":{
            "impacto":"int",
            "viabilidad":"int",
            "costo_beneficio":"int",
            "riesgo":"int",
            "alineacion":"int",
            "total":"float"
        },
        "racional_voto":"str",
        "banderas_riesgo":"list[str]",
        "valido":"bool"
    }

def compact_messages(batch):
    schema = json.dumps(schema_dict(), ensure_ascii=False)
    user = {
        "role":"user",
        "content": json.dumps({"schema": schema, "items": batch}, ensure_ascii=False)
    }
    return [{"role":"system","content":SYSTEM_FAST},
            {"role":"user","content":TEMPLATE_COMPACT.format(schema=schema)},
            user]

def call_llm(messages):
    if USE_MOCK:
        # Regla simple: categorizar por keywords y puntuar heurísticamente
        out = []
        for it in json.loads(messages[-1]["content"])["items"]:
            desc = (it.get("descripcion") or "").lower()
            cat = it.get("categoria_sugerida") or "Otro"
            if "solar" in desc or "energ" in desc:
                cat = "Educación"
            elif "huerto" in desc or "recicl" in desc or "compost" in desc:
                cat = "Sostenibilidad"
            elif "herramienta" in desc or "taller" in desc or "bicic" in desc:
                cat = "Tecnología"
            elif "biblioteca" in desc:
                cat = "Cultura"

            # Duración aproximada
            import re
            m = re.search(r"duración:\s*(\d+)\s*mes", it.get("descripcion"), re.I)
            dur = int(m.group(1)) if m else None

            # Scoring base
            impacto = 4 if "impacto" in desc or "emple" in desc else 3
            viabilidad = 4 if (it.get("monto_objetivo_eur",0) <= 30000) else 3
            costo_beneficio = 4 if it.get("monto_objetivo_eur",0) <= 20000 else 3
            riesgo = 3
            alineacion = 4
            total = round((impacto+viabilidad+costo_beneficio+riesgo+alineacion)/5,1)

            resumen = (it["descripcion"][:700] + "...") if len(it["descripcion"])>700 else it["descripcion"]
            out.append({
                "id": it["id"],
                "categoria": cat,
                "resumen_120w": resumen[:900],
                "duracion_meses": dur,
                "coste_eur": it.get("monto_objetivo_eur"),
                "riesgos_clave": ["vandalismo"] if "vandal" in desc else (["logística"] if "logíst" in desc else ["no especificado"]),
                "impacto_esperado": "beneficio social local y aprendizaje",
                "puntuacion": {
                    "impacto": impacto, "viabilidad": viabilidad,
                    "costo_beneficio": costo_beneficio, "riesgo": riesgo,
                    "alineacion": alineacion, "total": total
                },
                "racional_voto": "Alta alineación con la misión y costes razonables.",
                "banderas_riesgo": [] if dur is not None else ["duracion_no_especificada"],
                "valido": True
            })
        return out
    else:
        resp = openai.ChatCompletion.create(model=MODEL, messages=messages, temperature=0.2)
        txt = resp["choices"][0]["message"]["content"]
        return json.loads(txt)

def process_in_batches(items, batch_size=3):
    results = []
    for i in range(0, len(items), batch_size):
        batch = items[i:i+batch_size]
        msgs = compact_messages(batch)
        res = call_llm(msgs)
        if isinstance(res, list):
            results.extend(res)
        else:
            results.append(res)
    return results


## 3. Ejecutar la transformación

Procesamos en **lotes** para reducir tokens. La plantilla compacta instruye al modelo a devolver **solo JSON**.


In [ ]:

transformed = process_in_batches(PROPOSALS, batch_size=3)
len(transformed), transformed[0]


## 4. Validación ligera y ranking

In [ ]:

import pandas as pd
df = pd.DataFrame(transformed)
# Normalizar columna 'puntuacion.total'
df["score"] = df["puntuacion"].apply(lambda d: d.get("total", 0) if isinstance(d, dict) else 0.0)
df["coste_eur"] = df["coste_eur"].astype(float)
df_sorted = df.sort_values(["score","coste_eur"], ascending=[False, True]).reset_index(drop=True)
df_sorted[["id","categoria","duracion_meses","coste_eur","score"]]


## 5. Visualización simple (Top 5 por score)

In [ ]:

import matplotlib.pyplot as plt

top = df_sorted.head(5)
plt.figure()
plt.bar(top["id"], top["score"])
plt.title("Top 5 propuestas por score")
plt.xlabel("Propuesta")
plt.ylabel("Score")
plt.show()


## 6. Resultados por propuesta (JSON)

Se imprime una versión lista para guardar/consumir por el dashboard.


In [ ]:

print(json.dumps(transformed, ensure_ascii=False, indent=2)[:3000] + "\n...\n")


## 7. Coste estimado (aproximado)

A efectos didácticos, estimamos coste por **#propuestas × 1 llamada**. El coste real depende del modelo.


In [ ]:

N = len(PROPOSALS)
llamadas = math.ceil(N/3)
print(f"Propuestas: {N} | Llamadas esperadas: {llamadas} | Modo MOCK: {USE_MOCK}")


## 8. Conclusiones

- **Fast Prompting** permite comprimir instrucciones y reducir llamadas.  
- Salida **JSON validable** simplifica la integración con el dashboard de CivicVault.  
- Con un par de ajustes a las plantillas (few-shot mínimo, self-check), se logra consistencia sin sobrecoste.
